# Evaluation - Detailed Metrics and Analysis

This notebook evaluates the trained semantic segmentation model comprehensively, including multiple metrics at different thresholds and detailed visualizations.

## 1. Setup and Imports

In [ ]:
import sys
from pathlib import Path
import numpy as np
import cv2
from tqdm import tqdm
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, roc_curve, auc, precision_recall_curve, f1_score
import pandas as pd
import json
import warnings

warnings.filterwarnings('ignore')

# Add src to path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

from config import get_default_config
from data import create_dataloaders
from models import create_model
from metrics import SegmentationMetrics, compute_metrics
from utils import get_device, load_checkpoint

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')

print(f"PyTorch: {torch.__version__}")
print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

## 2. Configuration and Data Loading

In [ ]:
# Setup
config = get_default_config()
device = get_device(config.device.device)

project_root = Path.cwd().parent
models_dir = project_root / 'models' / 'checkpoints'
results_dir = project_root / 'results' / 'evaluation'
results_dir.mkdir(parents=True, exist_ok=True)

# Load data
print("Loading dataloaders...")
train_loader, val_loader, test_loader = create_dataloaders(config)

print(f"  Train batches: {len(train_loader)}")
print(f"  Val batches: {len(val_loader)}")
print(f"  Test batches: {len(test_loader)}")

## 3. Model Loading and Setup

In [ ]:
# Load model
model = create_model(
    model_type=config.model.model_type,
    in_channels=3,
    num_classes=1,
    **config.model.__dict__
)
model = model.to(device)

# Load checkpoint
checkpoints = list(models_dir.glob('*.pt'))
if checkpoints:
    checkpoint_path = checkpoints[-1]
    print(f"Loading: {checkpoint_path.name}")
    load_checkpoint(model, None, None, checkpoint_path, device)
    model.eval()
    print("Model loaded successfully")
else:
    print("WARNING: No checkpoint found")

## 4. Generate Predictions

In [ ]:
def get_predictions(model, dataloader, device, max_batches=None):
    """Generate predictions for entire dataset"""
    all_predictions = []
    all_targets = []
    
    model.eval()
    with torch.no_grad():
        for batch_idx, batch in enumerate(tqdm(dataloader, desc='Generating predictions')):
            if max_batches and batch_idx >= max_batches:
                break
            
            images = batch['image'].to(device)
            masks = batch['mask'].to(device)
            
            # Forward pass
            outputs = model(images)
            predictions = torch.sigmoid(outputs)
            
            all_predictions.append(predictions.cpu().numpy())
            all_targets.append(masks.cpu().numpy())
    
    predictions = np.concatenate(all_predictions, axis=0)
    targets = np.concatenate(all_targets, axis=0)
    
    return predictions, targets

print("Generating predictions on test set...")
pred_prob, targets = get_predictions(model, test_loader, device, max_batches=None)

print(f"Predictions shape: {pred_prob.shape}")
print(f"Targets shape: {targets.shape}")
print(f"Prediction range: [{pred_prob.min():.4f}, {pred_prob.max():.4f}]")

## 5. Multi-Threshold Metrics Evaluation

In [ ]:
# Evaluate at multiple thresholds
thresholds = [0.3, 0.4, 0.5, 0.6, 0.7]
threshold_results = {}

for threshold in thresholds:
    pred_binary = (pred_prob > threshold).astype(np.float32)
    
    # Compute metrics
    metrics = compute_metrics(
        predictions=torch.from_numpy(pred_binary),
        targets=torch.from_numpy(targets),
        threshold=threshold
    )
    
    threshold_results[threshold] = metrics
    
    print(f"\nThreshold: {threshold}")
    for metric_name, metric_value in metrics.items():
        print(f"  {metric_name}: {metric_value:.4f}")

## 6. Confusion Matrix Analysis

In [ ]:
# Confusion matrix at optimal threshold (0.5)
threshold = 0.5
pred_binary = (pred_prob > threshold).astype(np.int32).flatten()
targets_flat = targets.astype(np.int32).flatten()

cm = confusion_matrix(targets_flat, pred_binary)
tn, fp, fn, tp = cm.ravel()

print(f"Confusion Matrix (threshold={threshold}):")
print(f"  TP (True Positives): {tp:,}")
print(f"  TN (True Negatives): {tn:,}")
print(f"  FP (False Positives): {fp:,}")
print(f"  FN (False Negatives): {fn:,}")

# Visualize confusion matrix
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Heatmap
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
           xticklabels=['Background', 'Water'],
           yticklabels=['Background', 'Water'])
axes[0].set_title('Confusion Matrix', fontsize=12, fontweight='bold')
axes[0].set_ylabel('True Label', fontsize=11)
axes[0].set_xlabel('Predicted Label', fontsize=11)

# Normalized confusion matrix
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
sns.heatmap(cm_normalized, annot=True, fmt='.2%', cmap='Greens', ax=axes[1],
           xticklabels=['Background', 'Water'],
           yticklabels=['Background', 'Water'])
axes[1].set_title('Normalized Confusion Matrix', fontsize=12, fontweight='bold')
axes[1].set_ylabel('True Label', fontsize=11)
axes[1].set_xlabel('Predicted Label', fontsize=11)

plt.tight_layout()
plt.savefig(results_dir / 'confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nConfusion matrix visualization saved")

## 7. ROC and Precision-Recall Curves

In [ ]:
# Calculate curves
fpr, tpr, roc_thresholds = roc_curve(targets_flat, pred_prob.flatten())
roc_auc = auc(fpr, tpr)

precision, recall, pr_thresholds = precision_recall_curve(targets_flat, pred_prob.flatten())
pr_auc = auc(recall, precision)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC Curve
axes[0].plot(fpr, tpr, color='darkorange', lw=2.5, label=f'ROC curve (AUC = {roc_auc:.4f})')
axes[0].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', alpha=0.5, label='Random Classifier')
axes[0].set_xlim([0.0, 1.0])
axes[0].set_ylim([0.0, 1.05])
axes[0].set_xlabel('False Positive Rate', fontsize=11, fontweight='bold')
axes[0].set_ylabel('True Positive Rate', fontsize=11, fontweight='bold')
axes[0].set_title('ROC Curve', fontsize=12, fontweight='bold')
axes[0].legend(loc='lower right', fontsize=10)
axes[0].grid(True, alpha=0.3)

# Precision-Recall Curve
axes[1].plot(recall, precision, color='green', lw=2.5, label=f'PR curve (AUC = {pr_auc:.4f})')
axes[1].set_xlim([0.0, 1.0])
axes[1].set_ylim([0.0, 1.05])
axes[1].set_xlabel('Recall', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Precision', fontsize=11, fontweight='bold')
axes[1].set_title('Precision-Recall Curve', fontsize=12, fontweight='bold')
axes[1].legend(loc='best', fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(results_dir / 'roc_pr_curves.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"ROC AUC: {roc_auc:.4f}")
print(f"PR AUC: {pr_auc:.4f}")

## 8. Metrics Comparison Across Thresholds

In [ ]:
# Create metrics dataframe
metrics_data = []
for threshold, metrics in threshold_results.items():
    row = {'threshold': threshold}
    row.update(metrics)
    metrics_data.append(row)

metrics_df = pd.DataFrame(metrics_data)
print("\nMetrics at Different Thresholds:")
print(metrics_df.to_string(index=False))

# Visualize metrics
metrics_to_plot = ['iou', 'dice', 'precision', 'recall', 'f1']
metrics_to_plot = [m for m in metrics_to_plot if m in metrics_df.columns]

fig, axes = plt.subplots(len(metrics_to_plot), 1, figsize=(12, 3*len(metrics_to_plot)))

if len(metrics_to_plot) == 1:
    axes = [axes]

for idx, metric in enumerate(metrics_to_plot):
    axes[idx].plot(metrics_df['threshold'], metrics_df[metric], 
                   marker='o', linewidth=2.5, markersize=8, color='steelblue')
    axes[idx].fill_between(metrics_df['threshold'], metrics_df[metric], alpha=0.3)
    axes[idx].set_xlabel('Threshold', fontsize=11, fontweight='bold')
    axes[idx].set_ylabel(metric.upper(), fontsize=11, fontweight='bold')
    axes[idx].set_title(f'{metric.upper()} vs Threshold', fontsize=12, fontweight='bold')
    axes[idx].grid(True, alpha=0.3)
    axes[idx].set_ylim([0, 1])
    
    # Add value labels
    for x, y in zip(metrics_df['threshold'], metrics_df[metric]):
        axes[idx].text(x, y + 0.02, f'{y:.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig(results_dir / 'metrics_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nMetrics comparison visualization saved")

## 9. Per-Image Metrics Analysis

In [ ]:
# Calculate per-image metrics
threshold = 0.5
pred_binary = (pred_prob > threshold).astype(np.float32)

per_image_metrics = []

for img_idx in range(min(50, pred_binary.shape[0])):
    pred = pred_binary[img_idx].flatten()
    target = targets[img_idx].flatten()
    
    # Calculate metrics
    tp = np.sum((pred == 1) & (target == 1))
    fp = np.sum((pred == 1) & (target == 0))
    fn = np.sum((pred == 0) & (target == 1))
    tn = np.sum((pred == 0) & (target == 0))
    
    iou = tp / (tp + fp + fn + 1e-8)
    dice = (2 * tp) / (2 * tp + fp + fn + 1e-8)
    precision = tp / (tp + fp + 1e-8)
    recall = tp / (tp + fn + 1e-8)
    f1 = (2 * precision * recall) / (precision + recall + 1e-8)
    accuracy = (tp + tn) / (tp + tn + fp + fn + 1e-8)
    
    per_image_metrics.append({
        'image_idx': img_idx,
        'iou': iou,
        'dice': dice,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'accuracy': accuracy,
        'water_pixels': int(np.sum(target))
    })

per_image_df = pd.DataFrame(per_image_metrics)
print(f"\nPer-Image Metrics (first 10 images):")
print(per_image_df.head(10).to_string(index=False))

## 10. Metrics Distribution Visualization

In [ ]:
# Plot distributions
metrics_to_visualize = ['iou', 'dice', 'f1', 'accuracy']
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, metric in enumerate(metrics_to_visualize):
    data = per_image_df[metric].values
    
    axes[idx].hist(data, bins=20, color='steelblue', alpha=0.7, edgecolor='black')
    axes[idx].set_xlabel(metric.upper(), fontsize=11, fontweight='bold')
    axes[idx].set_ylabel('Frequency', fontsize=11, fontweight='bold')
    axes[idx].set_title(f'Distribution of {metric.upper()}', fontsize=12, fontweight='bold')
    axes[idx].grid(True, alpha=0.3, axis='y')
    
    # Add statistics
    stats_text = f"Mean: {data.mean():.3f}\nStd: {data.std():.3f}\nMin: {data.min():.3f}\nMax: {data.max():.3f}"
    axes[idx].text(0.98, 0.97, stats_text, transform=axes[idx].transAxes,
                  verticalalignment='top', horizontalalignment='right',
                  bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8),
                  fontsize=9, fontfamily='monospace')

plt.tight_layout()
plt.savefig(results_dir / 'metrics_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print("Metrics distribution visualization saved")

## 11. Pixel-Level Distribution Analysis

In [ ]:
# Analyze prediction confidence by pixel type
threshold = 0.5
pred_binary = (pred_prob > threshold).astype(np.float32)

# Separate predictions by ground truth
water_predictions = pred_prob[targets == 1]
background_predictions = pred_prob[targets == 0]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution of predictions for water pixels
axes[0].hist(water_predictions, bins=50, alpha=0.7, label='Actual Water', color='blue', edgecolor='black')
axes[0].hist(background_predictions, bins=50, alpha=0.7, label='Actual Background', color='brown', edgecolor='black')
axes[0].axvline(threshold, color='red', linestyle='--', linewidth=2.5, label=f'Threshold ({threshold})')
axes[0].set_xlabel('Predicted Probability', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Frequency', fontsize=11, fontweight='bold')
axes[0].set_title('Prediction Distribution by Ground Truth Class', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3, axis='y')

# Box plot
data_to_plot = [water_predictions, background_predictions]
axes[1].boxplot(data_to_plot, labels=['Water', 'Background'], patch_artist=True,
              boxprops=dict(facecolor='lightblue', alpha=0.7),
              medianprops=dict(color='red', linewidth=2),
              flierprops=dict(marker='o', markerfacecolor='red', markersize=5, alpha=0.5))
axes[1].axhline(threshold, color='red', linestyle='--', linewidth=2.5, label=f'Threshold ({threshold})')
axes[1].set_ylabel('Predicted Probability', fontsize=11, fontweight='bold')
axes[1].set_title('Prediction Confidence Distribution', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(results_dir / 'confidence_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nPrediction Statistics:")
print(f"  Water class - Mean: {water_predictions.mean():.4f}, Std: {water_predictions.std():.4f}")
print(f"  Background class - Mean: {background_predictions.mean():.4f}, Std: {background_predictions.std():.4f}")

## 12. Error Analysis - Worst Predictions

In [ ]:
# Sort by IoU to find worst predictions
worst_idx = per_image_df.nsmallest(5, 'iou')['image_idx'].values
best_idx = per_image_df.nlargest(5, 'iou')['image_idx'].values

fig, axes = plt.subplots(2, 5, figsize=(18, 8))

# Worst predictions
for idx, image_idx in enumerate(worst_idx):
    pred = pred_binary[image_idx, 0]
    target = targets[image_idx, 0]
    
    iou = per_image_df[per_image_df['image_idx'] == image_idx]['iou'].values[0]
    
    # Create error map
    error_map = np.zeros((pred.shape[0], pred.shape[1], 3), dtype=np.uint8)
    error_map[(pred == 1) & (target == 1)] = [0, 255, 0]  # TP - Green
    error_map[(pred == 1) & (target == 0)] = [0, 0, 255]  # FP - Red
    error_map[(pred == 0) & (target == 1)] = [255, 0, 0]  # FN - Blue
    error_map[(pred == 0) & (target == 0)] = [200, 200, 200]  # TN - Gray
    
    axes[0, idx].imshow(error_map)
    axes[0, idx].set_title(f'Worst #{idx+1}\nIoU: {iou:.3f}', fontsize=10, fontweight='bold')
    axes[0, idx].axis('off')

# Best predictions
for idx, image_idx in enumerate(best_idx):
    pred = pred_binary[image_idx, 0]
    target = targets[image_idx, 0]
    
    iou = per_image_df[per_image_df['image_idx'] == image_idx]['iou'].values[0]
    
    error_map = np.zeros((pred.shape[0], pred.shape[1], 3), dtype=np.uint8)
    error_map[(pred == 1) & (target == 1)] = [0, 255, 0]
    error_map[(pred == 1) & (target == 0)] = [0, 0, 255]
    error_map[(pred == 0) & (target == 1)] = [255, 0, 0]
    error_map[(pred == 0) & (target == 0)] = [200, 200, 200]
    
    axes[1, idx].imshow(error_map)
    axes[1, idx].set_title(f'Best #{idx+1}\nIoU: {iou:.3f}', fontsize=10, fontweight='bold')
    axes[1, idx].axis('off')

# Add legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor=[0/255, 255/255, 0/255], label='TP (Green)'),
    Patch(facecolor=[0/255, 0/255, 255/255], label='FP (Red)'),
    Patch(facecolor=[255/255, 0/255, 0/255], label='FN (Blue)'),
    Patch(facecolor=[200/255, 200/255, 200/255], label='TN (Gray)')
]
fig.legend(handles=legend_elements, loc='upper center', ncol=4, fontsize=10, bbox_to_anchor=(0.5, -0.02))

plt.tight_layout()
plt.savefig(results_dir / 'error_analysis_best_worst.png', dpi=150, bbox_inches='tight')
plt.show()

print("Error analysis visualization saved")

## 13. Create Evaluation Report

In [ ]:
# Create comprehensive evaluation report
threshold = 0.5
pred_binary = (pred_prob > threshold).astype(np.float32)

metrics_05 = threshold_results[0.5]

report = {
    'model_evaluation': {
        'model_type': config.model.model_type,
        'device': str(device),
        'threshold': threshold,
        'total_samples_evaluated': int(pred_binary.size)
    },
    'overall_metrics': {
        'iou': float(metrics_05.get('iou', 0)),
        'dice': float(metrics_05.get('dice', 0)),
        'precision': float(metrics_05.get('precision', 0)),
        'recall': float(metrics_05.get('recall', 0)),
        'f1': float(metrics_05.get('f1', 0)),
        'accuracy': float(metrics_05.get('accuracy', 0))
    },
    'roc_metrics': {
        'auc_roc': float(roc_auc),
        'auc_pr': float(pr_auc)
    },
    'confusion_matrix': {
        'true_positives': int(tp),
        'true_negatives': int(tn),
        'false_positives': int(fp),
        'false_negatives': int(fn)
    },
    'per_image_statistics': {
        'iou': {
            'mean': float(per_image_df['iou'].mean()),
            'std': float(per_image_df['iou'].std()),
            'min': float(per_image_df['iou'].min()),
            'max': float(per_image_df['iou'].max())
        },
        'dice': {
            'mean': float(per_image_df['dice'].mean()),
            'std': float(per_image_df['dice'].std()),
            'min': float(per_image_df['dice'].min()),
            'max': float(per_image_df['dice'].max())
        },
        'f1': {
            'mean': float(per_image_df['f1'].mean()),
            'std': float(per_image_df['f1'].std()),
            'min': float(per_image_df['f1'].min()),
            'max': float(per_image_df['f1'].max())
        }
    },
    'multi_threshold_results': {}
}

for thres, metrics in threshold_results.items():
    report['multi_threshold_results'][str(thres)] = {
        k: float(v) if isinstance(v, (int, float, np.floating)) else v
        for k, v in metrics.items()
    }

# Save report
report_path = results_dir / 'evaluation_report.json'
with open(report_path, 'w') as f:
    json.dump(report, f, indent=2)

print(f"Evaluation report saved to {report_path}")

# Also save per-image metrics
per_image_df.to_csv(results_dir / 'per_image_metrics.csv', index=False)
print(f"Per-image metrics saved to {results_dir / 'per_image_metrics.csv'}")

## 14. Summary Statistics

In [ ]:
print("\n" + "="*70)
print("EVALUATION SUMMARY")
print("="*70)

print(f"\nModel: {config.model.model_type}")
print(f"Device: {device}")
print(f"Threshold: {threshold}")
print(f"Total Samples: {pred_binary.size:,}")

print(f"\nOVERALL METRICS (Threshold={threshold}):")
print(f"  IoU: {metrics_05.get('iou', 0):.4f}")
print(f"  Dice: {metrics_05.get('dice', 0):.4f}")
print(f"  Precision: {metrics_05.get('precision', 0):.4f}")
print(f"  Recall: {metrics_05.get('recall', 0):.4f}")
print(f"  F1-Score: {metrics_05.get('f1', 0):.4f}")
print(f"  Accuracy: {metrics_05.get('accuracy', 0):.4f}")

print(f"\nROC METRICS:")
print(f"  AUC-ROC: {roc_auc:.4f}")
print(f"  AUC-PR: {pr_auc:.4f}")

print(f"\nCONFUSION MATRIX:")
print(f"  True Positives: {tp:,}")
print(f"  True Negatives: {tn:,}")
print(f"  False Positives: {fp:,}")
print(f"  False Negatives: {fn:,}")

print(f"\nPER-IMAGE STATISTICS (n={len(per_image_df)}):")
print(f"  IoU - Mean: {per_image_df['iou'].mean():.4f}, Std: {per_image_df['iou'].std():.4f}")
print(f"  Dice - Mean: {per_image_df['dice'].mean():.4f}, Std: {per_image_df['dice'].std():.4f}")
print(f"  F1 - Mean: {per_image_df['f1'].mean():.4f}, Std: {per_image_df['f1'].std():.4f}")
print(f"  Accuracy - Mean: {per_image_df['accuracy'].mean():.4f}, Std: {per_image_df['accuracy'].std():.4f}")

print(f"\nOUTPUT FILES:")
for file in sorted(results_dir.glob('*.png')):
    print(f"  - {file.name}")
for file in sorted(results_dir.glob('*.json')):
    print(f"  - {file.name}")
for file in sorted(results_dir.glob('*.csv')):
    print(f"  - {file.name}")

print(f"\nEvaluation completed successfully!")
print("="*70)